# Running Analytics - Data Exploration

**Date**: 2025-09-22  
**Focus**: Running activities analysis

This notebook explores running activities data from Strava, including:
1. First and last run dates
2. Weekly volume in kilometers
3. Weekly hours spent running
4. Basic feature exploration
5. Average heart rate per week when running

In [ ]:
import polars as pl
import pandas as pd  # Fallback for compatibility when needed
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.offline as pyo
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set Plotly to work in notebook
pyo.init_notebook_mode(connected=True)

# Polars configuration
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(15)

## Data Loading and Initial Exploration

In [ ]:
# Load the activities data using Polars
try:
    df = pl.read_csv('../data/01_raw/strava_activities.csv')
    print(f"Loaded {len(df)} activities")
    print(f"Data shape: {df.shape}")
    print(f"Columns: {df.columns}")
except FileNotFoundError:
    print("Data file not found. Please run the pipeline first:")
    print("kedro run --pipeline=day0 --params='max_activities=100'")
    df = pl.DataFrame()  # Empty dataframe for now

In [ ]:
if not df.is_empty():
    # Basic dataset information
    print("Dataset Overview:")
    print(f"Total activities: {len(df)}")
    print(f"Columns: {df.shape[1]}")
    print(f"Date range: {df['start_date'].min()} to {df['start_date'].max()}")
    
    # Activity types
    print("\nActivity types:")
    activity_counts = df['type'].value_counts().sort(pl.col('count'), descending=True)
    print(activity_counts)

## Filter for Running Activities

In [ ]:
if not df.is_empty():
    # Filter for running activities using Polars
    running_df = df.filter(pl.col('type') == 'Run')
    
    print(f"Total running activities: {len(running_df)}")
    
    if len(running_df) > 0:
        # Data processing with Polars expressions
        running_df = running_df.with_columns([
            # Convert start_date to datetime with proper format (ISO format with UTC)
            pl.col('start_date').str.to_datetime(format='%Y-%m-%dT%H:%M:%SZ').alias('start_date'),
            # Convert distance from meters to kilometers
            (pl.col('distance') / 1000).alias('distance_km'),
            # Convert moving_time from seconds to hours
            (pl.col('moving_time') / 3600).alias('moving_time_hours'),
            # Calculate pace (minutes per km)
            ((pl.col('moving_time') / 60) / (pl.col('distance') / 1000)).alias('pace_min_per_km')
        ]).with_columns([
            # Add week and year columns for aggregation
            pl.col('start_date').dt.week().alias('week'),
            pl.col('start_date').dt.year().alias('year')
        ]).with_columns([
            # Create year_week identifier
            (pl.col('year').cast(pl.String) + '-W' + pl.col('week').cast(pl.String).str.zfill(2)).alias('year_week')
        ])
        
        print("\nData preparation completed.")
        print(f"Date range: {running_df['start_date'].min()} to {running_df['start_date'].max()}")
        print(f"Distance range: {running_df['distance_km'].min():.2f} - {running_df['distance_km'].max():.2f} km")
        print(f"Duration range: {running_df['moving_time_hours'].min():.2f} - {running_df['moving_time_hours'].max():.2f} hours")
    else:
        print("No running activities found in the dataset.")
        running_df = pl.DataFrame()
else:
    running_df = pl.DataFrame()

## 1. First and Last Run Dates

In [ ]:
if not running_df.is_empty():
    # First and last run dates using Polars
    first_run = running_df['start_date'].min()
    last_run = running_df['start_date'].max()
    total_days = (last_run - first_run).total_seconds() / (24 * 3600)
    
    print(f"🏃‍♂️ RUNNING ACTIVITY TIMELINE")
    print(f"═" * 50)
    print(f"First run: {first_run.strftime('%B %d, %Y (%A)')}")
    print(f"Last run:  {last_run.strftime('%B %d, %Y (%A)')}")
    print(f"Total period: {total_days:.0f} days ({total_days/365.25:.1f} years)")
    print(f"Total runs: {len(running_df)}")
    print(f"Average runs per week: {len(running_df) / (total_days/7):.1f}")
    
    # Create interactive timeline visualization with Plotly
    running_sorted = running_df.sort('start_date')
    
    # Create the main scatter plot
    fig = px.scatter(
        running_sorted.to_pandas(),
        x='start_date', 
        y='distance_km',
        color='moving_time_hours',
        size='distance_km',
        title='Running Activities Timeline',
        labels={
            'start_date': 'Date',
            'distance_km': 'Distance (km)',
            'moving_time_hours': 'Duration (hours)'
        },
        color_continuous_scale='viridis',
        hover_data=['pace_min_per_km', 'moving_time_hours']
    )
    
    # Highlight first and last runs
    first_run_data = running_sorted.row(0, named=True)
    last_run_data = running_sorted.row(-1, named=True)
    
    fig.add_trace(go.Scatter(
        x=[first_run_data['start_date']],
        y=[first_run_data['distance_km']],
        mode='markers',
        marker=dict(size=15, color='green', symbol='circle', line=dict(width=2, color='darkgreen')),
        name=f'First Run ({first_run.strftime("%Y-%m-%d")})',
        hovertemplate='<b>First Run</b><br>Date: %{x}<br>Distance: %{y:.2f} km<extra></extra>'
    ))
    
    fig.add_trace(go.Scatter(
        x=[last_run_data['start_date']],
        y=[last_run_data['distance_km']],
        mode='markers',
        marker=dict(size=15, color='red', symbol='circle', line=dict(width=2, color='darkred')),
        name=f'Last Run ({last_run.strftime("%Y-%m-%d")})',
        hovertemplate='<b>Last Run</b><br>Date: %{x}<br>Distance: %{y:.2f} km<extra></extra>'
    ))
    
    fig.update_layout(
        height=600,
        showlegend=True,
        hovermode='closest'
    )
    
    fig.show()
else:
    print("No running data available for analysis.")

## 2. Weekly Volume in Kilometers

In [ ]:
if not running_df.is_empty():
    # Calculate weekly running volume using Polars
    weekly_volume = (
        running_df
        .group_by('year_week')
        .agg([
            pl.col('distance_km').sum(),
            pl.col('start_date').min().alias('week_start')
        ])
        .sort('week_start')
    )
    
    print(f"📊 WEEKLY RUNNING VOLUME")
    print(f"═" * 50)
    print(f"Average weekly volume: {weekly_volume['distance_km'].mean():.1f} km")
    print(f"Maximum weekly volume: {weekly_volume['distance_km'].max():.1f} km")
    print(f"Minimum weekly volume: {weekly_volume['distance_km'].min():.1f} km")
    print(f"Total weeks with running: {len(weekly_volume)}")
    
    # Create interactive weekly volume charts with Plotly
    weekly_volume_pd = weekly_volume.to_pandas()
    avg_distance = weekly_volume['distance_km'].mean()
    median_distance = weekly_volume['distance_km'].median()
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Weekly Running Volume Over Time', 'Distribution of Weekly Running Volume'),
        vertical_spacing=0.15
    )
    
    # Time series plot
    fig.add_trace(
        go.Scatter(
            x=weekly_volume_pd['week_start'],
            y=weekly_volume_pd['distance_km'],
            mode='lines+markers',
            name='Weekly Distance',
            line=dict(width=2),
            marker=dict(size=6),
            hovertemplate='<b>Week of %{x}</b><br>Distance: %{y:.1f} km<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Add average line
    fig.add_hline(
        y=avg_distance, 
        line_dash="dash", 
        line_color="red",
        annotation_text=f"Average: {avg_distance:.1f} km",
        row=1, col=1
    )
    
    # Distribution histogram
    fig.add_trace(
        go.Histogram(
            x=weekly_volume_pd['distance_km'],
            nbinsx=20,
            name='Distribution',
            opacity=0.7,
            hovertemplate='Distance Range: %{x}<br>Count: %{y}<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Add average and median lines to histogram
    fig.add_vline(
        x=avg_distance,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Avg: {avg_distance:.1f}",
        row=2, col=1
    )
    
    fig.add_vline(
        x=median_distance,
        line_dash="dash",
        line_color="orange",
        annotation_text=f"Median: {median_distance:.1f}",
        row=2, col=1
    )
    
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_yaxes(title_text="Weekly Distance (km)", row=1, col=1)
    fig.update_xaxes(title_text="Weekly Distance (km)", row=2, col=1)
    fig.update_yaxes(title_text="Frequency", row=2, col=1)
    
    fig.update_layout(height=800, showlegend=False)
    fig.show()
    
    # Show top 10 highest volume weeks
    print("\n🏆 TOP 10 HIGHEST VOLUME WEEKS:")
    top_weeks = weekly_volume.sort('distance_km', descending=True).head(10)
    for row in top_weeks.iter_rows(named=True):
        print(f"{row['year_week']}: {row['distance_km']:.1f} km (week of {row['week_start'].strftime('%B %d, %Y')})")
else:
    print("No running data available for weekly volume analysis.")

## 3. Weekly Hours Spent Running

In [ ]:
if not running_df.is_empty():
    # Calculate weekly running hours using Polars
    weekly_hours = (
        running_df
        .group_by('year_week')
        .agg([
            pl.col('moving_time_hours').sum(),
            pl.col('start_date').min().alias('week_start'),
            pl.col('distance_km').sum().alias('total_distance')
        ])
        .with_columns([
            # Calculate average pace per week
            ((pl.col('moving_time_hours') * 60) / pl.col('total_distance')).alias('avg_pace_min_per_km')
        ])
        .sort('week_start')
    )
    
    print(f"⏱️ WEEKLY RUNNING HOURS")
    print(f"═" * 50)
    print(f"Average weekly hours: {weekly_hours['moving_time_hours'].mean():.1f} hours")
    print(f"Maximum weekly hours: {weekly_hours['moving_time_hours'].max():.1f} hours")
    print(f"Minimum weekly hours: {weekly_hours['moving_time_hours'].min():.1f} hours")
    print(f"Total running time: {weekly_hours['moving_time_hours'].sum():.1f} hours ({weekly_hours['moving_time_hours'].sum()/24:.1f} days)")
    
    # Convert to pandas for plotting
    weekly_hours_pd = weekly_hours.to_pandas()
    avg_hours = weekly_hours['moving_time_hours'].mean()
    
    # Create interactive charts with Plotly
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Weekly Running Hours Over Time', 'Relationship between Weekly Distance and Time'),
        vertical_spacing=0.15
    )
    
    # Time series plot
    fig.add_trace(
        go.Scatter(
            x=weekly_hours_pd['week_start'],
            y=weekly_hours_pd['moving_time_hours'],
            mode='lines+markers',
            name='Weekly Hours',
            line=dict(width=2, color='orange'),
            marker=dict(size=6, color='orange'),
            hovertemplate='<b>Week of %{x}</b><br>Hours: %{y:.1f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Add average line
    fig.add_hline(
        y=avg_hours,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Average: {avg_hours:.1f} hours",
        row=1, col=1
    )
    
    # Correlation scatter plot
    fig.add_trace(
        go.Scatter(
            x=weekly_hours_pd['total_distance'],
            y=weekly_hours_pd['moving_time_hours'],
            mode='markers',
            marker=dict(
                size=8,
                color=weekly_hours_pd['avg_pace_min_per_km'],
                colorscale='plasma',
                showscale=True,
                colorbar=dict(title="Avg Pace (min/km)", x=1.02)
            ),
            text=weekly_hours_pd['year_week'],
            hovertemplate='<b>%{text}</b><br>Distance: %{x:.1f} km<br>Hours: %{y:.1f}<br>Avg Pace: %{marker.color:.1f} min/km<extra></extra>',
            name='Weekly Data'
        ),
        row=2, col=1
    )
    
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_yaxes(title_text="Weekly Hours", row=1, col=1)
    fig.update_xaxes(title_text="Weekly Distance (km)", row=2, col=1)
    fig.update_yaxes(title_text="Weekly Hours", row=2, col=1)
    
    fig.update_layout(height=800, showlegend=False)
    fig.show()
    
    # Show correlation coefficient
    correlation = weekly_hours_pd['total_distance'].corr(weekly_hours_pd['moving_time_hours'])
    print(f"\n📈 Correlation between weekly distance and time: {correlation:.3f}")
    
    # Show top 10 highest hour weeks
    print("\n🏆 TOP 10 HIGHEST HOUR WEEKS:")
    top_hour_weeks = weekly_hours.sort('moving_time_hours', descending=True).head(10)
    for row in top_hour_weeks.iter_rows(named=True):
        print(f"{row['year_week']}: {row['moving_time_hours']:.1f} hours, {row['total_distance']:.1f} km (avg pace: {row['avg_pace_min_per_km']:.1f} min/km)")
else:
    print("No running data available for weekly hours analysis.")

## 4. Basic Feature Exploration

In [ ]:
if not running_df.is_empty():
    print(f"🔍 RUNNING ACTIVITY FEATURES")
    print(f"═" * 50)
    
    # Key running metrics
    key_features = ['distance_km', 'moving_time_hours', 'pace_min_per_km', 
                   'average_speed', 'total_elevation_gain', 'average_heartrate']
    
    # Basic statistics using Polars
    print("BASIC STATISTICS:")
    stats_df = running_df.select(key_features).describe()
    print(stats_df)
    
    # Create interactive feature exploration plots with Plotly
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            'Distribution of Run Distances', 'Distribution of Running Pace',
            'Distance vs Pace Relationship', 'Distribution of Elevation Gain',
            'Running Frequency by Day of Week', 'Running Frequency by Month'
        ),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]],
        vertical_spacing=0.08,
        horizontal_spacing=0.1
    )
    
    # 1. Distance distribution
    fig.add_trace(
        go.Histogram(
            x=running_df['distance_km'].to_pandas(),
            nbinsx=30,
            name='Distance Distribution',
            opacity=0.7,
            hovertemplate='Distance: %{x:.1f} km<br>Count: %{y}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # 2. Pace distribution (filter reasonable values)
    pace_filtered = running_df.filter(
        (pl.col('pace_min_per_km') > 3) & (pl.col('pace_min_per_km') < 10)
    )['pace_min_per_km'].to_pandas()
    
    fig.add_trace(
        go.Histogram(
            x=pace_filtered,
            nbinsx=30,
            name='Pace Distribution',
            opacity=0.7,
            marker_color='orange',
            hovertemplate='Pace: %{x:.1f} min/km<br>Count: %{y}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # 3. Distance vs Pace scatter
    pace_distance_data = running_df.filter(
        (pl.col('pace_min_per_km') > 3) & (pl.col('pace_min_per_km') < 10)
    ).to_pandas()
    
    fig.add_trace(
        go.Scatter(
            x=pace_distance_data['distance_km'],
            y=pace_distance_data['pace_min_per_km'],
            mode='markers',
            name='Distance vs Pace',
            opacity=0.6,
            hovertemplate='Distance: %{x:.1f} km<br>Pace: %{y:.1f} min/km<extra></extra>'
        ),
        row=2, col=1
    )
    
    # 4. Elevation gain distribution
    elev_data = running_df.filter(pl.col('total_elevation_gain').is_not_null())
    if not elev_data.is_empty():
        fig.add_trace(
            go.Histogram(
                x=elev_data['total_elevation_gain'].to_pandas(),
                nbinsx=30,
                name='Elevation Distribution',
                opacity=0.7,
                marker_color='green',
                hovertemplate='Elevation: %{x:.0f} m<br>Count: %{y}<extra></extra>'
            ),
            row=2, col=2
        )
    
    # 5. Running activities by day of week using Polars
    day_counts = (
        running_df
        .with_columns(pl.col('start_date').dt.strftime('%A').alias('day_of_week'))
        .group_by('day_of_week')
        .len()
        .sort('len', descending=True)
    )
    
    # Reorder by week day
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_counts_ordered = []
    for day in day_order:
        count = day_counts.filter(pl.col('day_of_week') == day)['len']
        day_counts_ordered.append(count[0] if len(count) > 0 else 0)
    
    fig.add_trace(
        go.Bar(
            x=day_order,
            y=day_counts_ordered,
            name='Day of Week',
            opacity=0.7,
            marker_color='purple',
            hovertemplate='%{x}<br>Runs: %{y}<extra></extra>'
        ),
        row=3, col=1
    )
    
    # 6. Monthly running pattern using Polars
    month_counts = (
        running_df
        .with_columns(pl.col('start_date').dt.strftime('%B').alias('month'))
        .group_by('month')
        .len()
        .sort('len', descending=True)
    )
    
    # Reorder by month
    month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                  'July', 'August', 'September', 'October', 'November', 'December']
    month_counts_ordered = []
    for month in month_order:
        count = month_counts.filter(pl.col('month') == month)['len']
        month_counts_ordered.append(count[0] if len(count) > 0 else 0)
    
    fig.add_trace(
        go.Bar(
            x=month_order,
            y=month_counts_ordered,
            name='Month',
            opacity=0.7,
            marker_color='red',
            hovertemplate='%{x}<br>Runs: %{y}<extra></extra>'
        ),
        row=3, col=2
    )
    
    # Update layout
    fig.update_xaxes(title_text="Distance (km)", row=1, col=1)
    fig.update_yaxes(title_text="Frequency", row=1, col=1)
    fig.update_xaxes(title_text="Pace (min/km)", row=1, col=2)
    fig.update_yaxes(title_text="Frequency", row=1, col=2)
    fig.update_xaxes(title_text="Distance (km)", row=2, col=1)
    fig.update_yaxes(title_text="Pace (min/km)", row=2, col=1)
    fig.update_xaxes(title_text="Elevation Gain (m)", row=2, col=2)
    fig.update_yaxes(title_text="Frequency", row=2, col=2)
    fig.update_xaxes(title_text="Day of Week", row=3, col=1)
    fig.update_yaxes(title_text="Number of Runs", row=3, col=1)
    fig.update_xaxes(title_text="Month", row=3, col=2)
    fig.update_yaxes(title_text="Number of Runs", row=3, col=2)
    
    fig.update_layout(height=1200, showlegend=False)
    fig.show()
    
    # Additional insights using Polars aggregations
    most_active_day = day_counts.row(0, named=True)
    least_active_day = day_counts.sort('len').row(0, named=True)
    most_active_month = month_counts.row(0, named=True)
    
    print(f"\n📋 ADDITIONAL INSIGHTS:")
    print(f"Most active day: {most_active_day['day_of_week']} ({most_active_day['len']} runs)")
    print(f"Least active day: {least_active_day['day_of_week']} ({least_active_day['len']} runs)")
    print(f"Most active month: {most_active_month['month']} ({most_active_month['len']} runs)")
    print(f"Average distance: {running_df['distance_km'].mean():.2f} km")
    print(f"Longest run: {running_df['distance_km'].max():.2f} km")
    print(f"Shortest run: {running_df['distance_km'].min():.2f} km")
    
    if not elev_data.is_empty():
        print(f"Average elevation gain: {elev_data['total_elevation_gain'].mean():.0f} m")
        print(f"Maximum elevation gain: {elev_data['total_elevation_gain'].max():.0f} m")
else:
    print("No running data available for feature exploration.")

## 5. Average Heart Rate per Week When Running

In [ ]:
if not running_df.is_empty():
    # Filter for runs with heart rate data using Polars
    hr_data = running_df.filter(
        pl.col('average_heartrate').is_not_null() & 
        (pl.col('average_heartrate') > 0)
    )
    
    if not hr_data.is_empty():
        # Calculate weekly average heart rate using Polars
        weekly_hr = (
            hr_data
            .group_by('year_week')
            .agg([
                pl.col('average_heartrate').mean().alias('avg_hr'),
                pl.col('average_heartrate').min().alias('min_hr'),
                pl.col('average_heartrate').max().alias('max_hr'),
                pl.col('average_heartrate').len().alias('run_count'),
                pl.col('start_date').min().alias('week_start'),
                pl.col('distance_km').sum().alias('total_distance'),
                pl.col('moving_time_hours').sum().alias('total_hours')
            ])
            .sort('week_start')
        )
        
        print(f"❤️ WEEKLY HEART RATE ANALYSIS")
        print(f"═" * 50)
        print(f"Runs with heart rate data: {len(hr_data)} / {len(running_df)} ({len(hr_data)/len(running_df)*100:.1f}%)")
        print(f"Average heart rate across all runs: {hr_data['average_heartrate'].mean():.0f} bpm")
        print(f"Heart rate range: {hr_data['average_heartrate'].min():.0f} - {hr_data['average_heartrate'].max():.0f} bpm")
        print(f"Weeks with heart rate data: {len(weekly_hr)}")
        
        # Convert to pandas for plotting
        weekly_hr_pd = weekly_hr.to_pandas()
        hr_data_pd = hr_data.to_pandas()
        
        # Create interactive heart rate analysis plots with Plotly
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Weekly Average Heart Rate Over Time',
                'Distribution of Average Heart Rate',
                'Heart Rate vs Pace Relationship',
                'Weekly HR vs Training Load'
            ),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]],
            vertical_spacing=0.12,
            horizontal_spacing=0.1
        )
        
        # 1. Weekly average heart rate over time
        avg_hr_overall = hr_data['average_heartrate'].mean()
        
        fig.add_trace(
            go.Scatter(
                x=weekly_hr_pd['week_start'],
                y=weekly_hr_pd['avg_hr'],
                mode='lines+markers',
                name='Weekly Avg HR',
                line=dict(width=2, color='red'),
                marker=dict(size=6),
                hovertemplate='<b>Week of %{x}</b><br>Avg HR: %{y:.0f} bpm<br>Runs: %{customdata}<extra></extra>',
                customdata=weekly_hr_pd['run_count']
            ),
            row=1, col=1
        )
        
        # Add HR range as fill
        fig.add_trace(
            go.Scatter(
                x=weekly_hr_pd['week_start'],
                y=weekly_hr_pd['max_hr'],
                mode='lines',
                line=dict(width=0),
                showlegend=False,
                hoverinfo='skip'
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=weekly_hr_pd['week_start'],
                y=weekly_hr_pd['min_hr'],
                mode='lines',
                line=dict(width=0),
                fillcolor='rgba(255, 0, 0, 0.2)',
                fill='tonexty',
                showlegend=False,
                hoverinfo='skip'
            ),
            row=1, col=1
        )
        
        # Add overall average line
        fig.add_hline(
            y=avg_hr_overall,
            line_dash="dash",
            line_color="darkred",
            annotation_text=f"Overall Avg: {avg_hr_overall:.0f} bpm",
            row=1, col=1
        )
        
        # 2. Heart rate distribution
        fig.add_trace(
            go.Histogram(
                x=hr_data_pd['average_heartrate'],
                nbinsx=30,
                name='HR Distribution',
                opacity=0.7,
                marker_color='red',
                hovertemplate='HR Range: %{x}<br>Count: %{y}<extra></extra>'
            ),
            row=1, col=2
        )
        
        # Add mean and median lines
        fig.add_vline(
            x=avg_hr_overall,
            line_dash="dash",
            line_color="darkred",
            annotation_text=f"Mean: {avg_hr_overall:.0f}",
            row=1, col=2
        )
        
        median_hr = hr_data['average_heartrate'].median()
        fig.add_vline(
            x=median_hr,
            line_dash="dash",
            line_color="orange",
            annotation_text=f"Median: {median_hr:.0f}",
            row=1, col=2
        )
        
        # 3. Heart rate vs pace relationship
        hr_pace_data = hr_data.filter(
            (pl.col('pace_min_per_km') > 3) & (pl.col('pace_min_per_km') < 10)
        ).to_pandas()
        
        if not hr_pace_data.empty:
            fig.add_trace(
                go.Scatter(
                    x=hr_pace_data['pace_min_per_km'],
                    y=hr_pace_data['average_heartrate'],
                    mode='markers',
                    marker=dict(
                        size=6,
                        color=hr_pace_data['distance_km'],
                        colorscale='viridis',
                        showscale=True,
                        colorbar=dict(title="Distance (km)", x=0.48, y=0.25, len=0.4)
                    ),
                    name='HR vs Pace',
                    hovertemplate='<b>Pace:</b> %{x:.1f} min/km<br><b>HR:</b> %{y:.0f} bpm<br><b>Distance:</b> %{marker.color:.1f} km<extra></extra>'
                ),
                row=2, col=1
            )
            
            # Calculate and show correlation
            hr_pace_corr = hr_pace_data['pace_min_per_km'].corr(hr_pace_data['average_heartrate'])
            fig.add_annotation(
                text=f'Correlation: {hr_pace_corr:.3f}',
                x=0.25, y=0.35,
                xref='paper', yref='paper',
                showarrow=False,
                bgcolor="white",
                bordercolor="black",
                borderwidth=1
            )
        
        # 4. Weekly heart rate vs training load
        fig.add_trace(
            go.Scatter(
                x=weekly_hr_pd['total_distance'],
                y=weekly_hr_pd['avg_hr'],
                mode='markers',
                marker=dict(
                    size=weekly_hr_pd['run_count'] * 3,
                    color=weekly_hr_pd['total_hours'],
                    colorscale='plasma',
                    showscale=True,
                    colorbar=dict(title="Weekly Hours", x=1.02, y=0.25, len=0.4)
                ),
                text=weekly_hr_pd['year_week'],
                name='Weekly HR vs Load',
                hovertemplate='<b>%{text}</b><br>Distance: %{x:.1f} km<br>Avg HR: %{y:.0f} bpm<br>Runs: %{customdata}<br>Hours: %{marker.color:.1f}<extra></extra>',
                customdata=weekly_hr_pd['run_count']
            ),
            row=2, col=2
        )
        
        # Update axes labels
        fig.update_xaxes(title_text="Date", row=1, col=1)
        fig.update_yaxes(title_text="Heart Rate (bpm)", row=1, col=1)
        fig.update_xaxes(title_text="Average Heart Rate (bpm)", row=1, col=2)
        fig.update_yaxes(title_text="Frequency", row=1, col=2)
        fig.update_xaxes(title_text="Pace (min/km)", row=2, col=1)
        fig.update_yaxes(title_text="Average Heart Rate (bpm)", row=2, col=1)
        fig.update_xaxes(title_text="Weekly Distance (km)", row=2, col=2)
        fig.update_yaxes(title_text="Average Heart Rate (bpm)", row=2, col=2)
        
        fig.update_layout(height=800, showlegend=False)
        fig.show()
        
        # Heart rate zones analysis
        print(f"\n💓 HEART RATE ZONES ANALYSIS:")
        max_hr_est = 190  # Rough estimate, user can adjust
        
        # Define HR zones as percentages of max HR
        zones = {
            'Zone 1 (50-60%)': (max_hr_est * 0.5, max_hr_est * 0.6),
            'Zone 2 (60-70%)': (max_hr_est * 0.6, max_hr_est * 0.7),
            'Zone 3 (70-80%)': (max_hr_est * 0.7, max_hr_est * 0.8),
            'Zone 4 (80-90%)': (max_hr_est * 0.8, max_hr_est * 0.9),
            'Zone 5 (90%+)': (max_hr_est * 0.9, 250)
        }
        
        print(f"(Assuming max HR of {max_hr_est} bpm - adjust as needed)")
        for zone, (low, high) in zones.items():
            count = len(hr_data.filter(
                (pl.col('average_heartrate') >= low) & 
                (pl.col('average_heartrate') < high)
            ))
            percentage = count / len(hr_data) * 100
            print(f"{zone}: {count} runs ({percentage:.1f}%) - {low:.0f}-{high:.0f} bpm")
        
        # Show weeks with highest and lowest average HR
        print(f"\n🏆 WEEKS WITH HIGHEST AVERAGE HEART RATE:")
        top_hr_weeks = weekly_hr.sort('avg_hr', descending=True).head(5)
        for row in top_hr_weeks.iter_rows(named=True):
            print(f"{row['year_week']}: {row['avg_hr']:.0f} bpm (from {row['run_count']} runs, {row['total_distance']:.1f} km)")
        
        print(f"\n🔽 WEEKS WITH LOWEST AVERAGE HEART RATE:")
        low_hr_weeks = weekly_hr.sort('avg_hr').head(5)
        for row in low_hr_weeks.iter_rows(named=True):
            print(f"{row['year_week']}: {row['avg_hr']:.0f} bpm (from {row['run_count']} runs, {row['total_distance']:.1f} km)")
        
    else:
        print(f"❤️ No heart rate data available in the running activities.")
        print("Heart rate data might be missing or set to 0/null in the dataset.")
else:
    print("No running data available for heart rate analysis.")

## Summary and Conclusions

In [ ]:
if not running_df.is_empty():
    print(f"🏃‍♂️ RUNNING ANALYTICS SUMMARY")
    print(f"═" * 60)
    
    # Overall statistics using Polars
    total_distance = running_df['distance_km'].sum()
    total_time = running_df['moving_time_hours'].sum()
    avg_pace = ((running_df['moving_time'] / 60) / running_df['distance_km']).mean()
    
    print(f"📊 OVERALL STATISTICS:")
    print(f"   Total runs: {len(running_df)}")
    print(f"   Total distance: {total_distance:.1f} km ({total_distance/1.60934:.1f} miles)")
    print(f"   Total time: {total_time:.1f} hours ({total_time/24:.1f} days)")
    print(f"   Average pace: {avg_pace:.2f} min/km")
    print(f"   Average distance per run: {running_df['distance_km'].mean():.2f} km")
    print(f"   Average duration per run: {running_df['moving_time_hours'].mean():.2f} hours")
    
    # Time period analysis
    if len(running_df) > 1:
        first_run = running_df['start_date'].min()
        last_run = running_df['start_date'].max()
        period_days = (last_run - first_run).total_seconds() / (24 * 3600) + 1
        
        print(f"\n📅 TIME PERIOD ANALYSIS:")
        print(f"   Period: {first_run.strftime('%Y-%m-%d')} to {last_run.strftime('%Y-%m-%d')}")
        print(f"   Duration: {period_days:.0f} days ({period_days/365.25:.1f} years)")
        print(f"   Runs per week: {len(running_df) / (period_days/7):.1f}")
        print(f"   Distance per week: {total_distance / (period_days/7):.1f} km")
        print(f"   Hours per week: {total_time / (period_days/7):.1f} hours")
    
    # Heart rate summary
    hr_data = running_df.filter(
        pl.col('average_heartrate').is_not_null() & 
        (pl.col('average_heartrate') > 0)
    )
    
    if not hr_data.is_empty():
        print(f"\n❤️ HEART RATE SUMMARY:")
        print(f"   Runs with HR data: {len(hr_data)} / {len(running_df)} ({len(hr_data)/len(running_df)*100:.1f}%)")
        print(f"   Average heart rate: {hr_data['average_heartrate'].mean():.0f} bpm")
        print(f"   HR range: {hr_data['average_heartrate'].min():.0f} - {hr_data['average_heartrate'].max():.0f} bpm")
    
    # Personal records
    print(f"\n🏆 PERSONAL RECORDS:")
    print(f"   Longest run: {running_df['distance_km'].max():.2f} km")
    fastest_pace = running_df.filter(
        (pl.col('pace_min_per_km') > 3) & (pl.col('pace_min_per_km') < 10)
    )['pace_min_per_km'].min()
    print(f"   Fastest pace: {fastest_pace:.2f} min/km")
    
    elev_data = running_df.filter(pl.col('total_elevation_gain').is_not_null())
    if not elev_data.is_empty():
        print(f"   Most elevation: {elev_data['total_elevation_gain'].max():.0f} m")
    
    # Recommendations
    print(f"\n💡 INSIGHTS & RECOMMENDATIONS:")
    
    # Weekly consistency (calculate if we have weekly data)
    if 'weekly_volume' in locals():
        weekly_vol_pd = weekly_volume.to_pandas()
        cv_volume = weekly_vol_pd['distance_km'].std() / weekly_vol_pd['distance_km'].mean()
        if cv_volume < 0.3:
            print(f"   ✅ Consistent weekly volume (CV: {cv_volume:.2f})")
        else:
            print(f"   ⚠️ Variable weekly volume (CV: {cv_volume:.2f}) - consider more consistent training")
    
    # Training frequency
    if len(running_df) > 1:
        first_run = running_df['start_date'].min()
        last_run = running_df['start_date'].max()
        period_days = (last_run - first_run).total_seconds() / (24 * 3600) + 1
        
        if period_days > 30:
            runs_per_week = len(running_df) / (period_days/7)
            if runs_per_week >= 3:
                print(f"   ✅ Good training frequency ({runs_per_week:.1f} runs/week)")
            else:
                print(f"   💪 Consider increasing frequency (currently {runs_per_week:.1f} runs/week)")
    
    # Heart rate zones
    if not hr_data.is_empty():
        max_hr_est = 190
        easy_pace_runs = len(hr_data.filter(pl.col('average_heartrate') < max_hr_est * 0.7))
        easy_pace_percentage = easy_pace_runs / len(hr_data) * 100
        
        if easy_pace_percentage >= 80:
            print(f"   ✅ Good aerobic base ({easy_pace_percentage:.0f}% easy pace)")
        else:
            print(f"   📈 Consider more easy pace runs (currently {easy_pace_percentage:.0f}%)")
    
    print(f"\n" + "═" * 60)
    print(f"Analysis completed on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Data source: ../data/01_raw/strava_activities.csv")
    print(f"Powered by Polars for fast data processing and Plotly for interactive visualizations")
    
else:
    print("❌ No running data available for analysis.")
    print("\nTo get started:")
    print("1. Run: kedro run --pipeline=day0 --params='max_activities=100'")
    print("2. Then re-run this notebook")